# IndoBERT Clickbait Detection — Complete Pipeline

This notebook runs the complete training and evaluation pipeline for IndoBERT-based clickbait detection with robustness evaluation.

**Requirements:**
- GPU Runtime (T4 for testing, A100 for full training)
- 5 CSV files with columns: `id`, `source`, `date`, `title`, `Label`, `url`
  (one file per domain: technology.csv, politic.csv, health.csv, sport.csv, education.csv)
- ~(TBA) hours for complete execution

**Intermediate-state cells** are included throughout so you can inspect the
data at every major transformation step:
- Raw CSV → after loading
- After stratified split (train / val / test sizes + class distributions)
- Sample texts before tokenisation
- Token IDs, attention masks, decoded tokens
- Before / after perturbation (low / medium / high)
- In-domain evaluation inputs and outputs
- Cross-domain evaluation inputs and predictions

## 1. Environment Setup

In [1]:
# Check GPU availability
!nvidia-smi

Wed Jul 29 04:01:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |                  N/A |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Clone repository from GitHub — the dataset CSVs live inside the repo,
# so cloning is the only setup step needed; no manual CSV uploads required.
!git clone https://github.com/gredss/domain-shift-trustworthy-ai.git
%cd domain-shift-trustworthy-ai

# Confirm the five domain CSV files arrived with the clone
!ls -lh dataset/data/*.csv

In [ ]:
# Install dependencies
!pip install -q torch transformers pandas numpy scikit-learn scipy streamlit plotly tqdm

## 2. Data Preparation

In [ ]:
# Create output directories on Drive (CSVs come from the cloned repo — no dataset dir needed here)
!mkdir -p "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints"
!mkdir -p "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output"
!mkdir -p "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results"

### 2a. Inspect Raw CSVs (before any processing)

Verify the five domain files loaded correctly: row counts, columns, and raw label distributions.

In [ ]:
# ── INPUT: raw CSV files as loaded from disk ──────────────────────────────────
import pandas as pd, glob

DATA_DIR = "/content/domain-shift-trustworthy-ai/dataset/data"
csv_files = sorted(glob.glob(f"{DATA_DIR}/*.csv"))
print(f"Found {len(csv_files)} CSV files in {DATA_DIR}\n")
print(f"{'Domain':<14} {'Rows':>6}  {'Label-0':>8}  {'Label-1':>8}  {'CB%':>6}  {'Avg title len':>14}")
print("-" * 62)

all_dfs = []
for path in csv_files:
    domain = path.split("/")[-1].replace(".csv", "").capitalize()
    df = pd.read_csv(path)
    c0 = (df["Label"] == 0).sum()
    c1 = (df["Label"] == 1).sum()
    cb_pct = c1 / len(df) * 100
    avg_len = df["title"].str.len().mean()
    print(f"{domain:<14} {len(df):>6}  {c0:>8}  {c1:>8}  {cb_pct:>5.1f}%  {avg_len:>14.1f}")
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
print("-" * 62)
print(f"{'TOTAL':<14} {len(combined):>6}  {(combined['Label']==0).sum():>8}  {(combined['Label']==1).sum():>8}")

print("\n── 3 sample rows from the first CSV ──")
display(pd.read_csv(csv_files[0]).head(3))

### 2b. Inspect Train / Val / Test Splits (after stratified split)

Reproduces the exact 70 / 15 / 15 stratified split that `train_pipeline.py` will use,
so you can verify sizes and class balance **before** training starts.

In [ ]:
# ── OUTPUT of data_manager.stratified_split_by_domain() ──────────────────────
import sys, os
sys.path.insert(0, "/content/domain-shift-trustworthy-ai/src")

from data_manager import DataManager

dm = DataManager.from_dataset_directory(
    dataset_dir=DATA_DIR,
    tokenizer_name="indobenchmark/indobert-base-p1",
    max_length=128,
    random_seed=42
)

domain_splits = dm.stratified_split_by_domain(train_size=0.70, val_size=0.15, test_size=0.15)

print(f"{'Domain':<14} {'Train':>7} {'Val':>5} {'Test':>5}  "
      f"{'Train CB%':>10}  {'Val CB%':>8}  {'Test CB%':>9}")
print("-" * 66)
for domain, splits in domain_splits.items():
    tr, va, te = splits["train"], splits["val"], splits["test"]
    def cb(df): return df["label"].mean() * 100
    print(f"{domain:<14} {len(tr):>7} {len(va):>5} {len(te):>5}  "
          f"{cb(tr):>9.1f}%  {cb(va):>7.1f}%  {cb(te):>8.1f}%")

print("\n── 5 sample training rows from Education split ──")
display(domain_splits["Education"]["train"][["text", "label", "domain"]].head(5))

### 2c. Inspect Tokenisation (before → after)

Shows how raw headline text is converted to token IDs and attention masks
that the model actually receives. Compare the original string with the decoded tokens.

In [ ]:
# ── INPUT: raw text strings  |  OUTPUT: token IDs, attention masks, decoded ──
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")
MAX_LEN = 128

# Pick 4 representative examples: 2 clickbait (label=1), 2 non-clickbait (label=0)
edu_train = domain_splits["Education"]["train"]
samples = pd.concat([
    edu_train[edu_train["label"] == 1].head(2),
    edu_train[edu_train["label"] == 0].head(2)
]).reset_index(drop=True)

SEP = "─" * 80
for _, row in samples.iterrows():
    text  = str(row["text"])
    label = "clickbait" if row["label"] == 1 else "non-clickbait"
    enc   = tokenizer(text, max_length=MAX_LEN, padding="max_length",
                      truncation=True, return_tensors="np")
    ids   = enc["input_ids"][0]
    mask  = enc["attention_mask"][0]
    n_real = int(mask.sum())
    decoded = tokenizer.decode(ids[:n_real], skip_special_tokens=False)

    print(SEP)
    print(f"label      : {label}")
    print(f"BEFORE     : {text}")
    print(f"input_ids  : {ids[:12].tolist()} … (seq_len={MAX_LEN}, real_tokens={n_real}, pad={MAX_LEN-n_real})")
    print(f"attn_mask  : {mask[:12].tolist()} …")
    print(f"AFTER      : {decoded}")
print(SEP)

## 3. Model Training

Train IndoBERT models. Start with base model for testing, then scale to all models.

> `--debug` is passed so the pipeline prints intermediate state to this cell's
> output: split sizes confirmed at load time, per-batch logits & loss every
> 50 batches, and epoch-level train/val metrics.

In [ ]:
# Train base model (recommended for initial testing)
# Trains one specialist model per domain (5 specialists x 1 variant = 5 models).
# --dataset-dir points to the CSV files inside the cloned GitHub repo.
# --debug prints intermediate state: split confirmation, batch logits, epoch metrics.
!python src/train_pipeline.py \
    --model base \
    --dataset-dir /content/domain-shift-trustworthy-ai/dataset/data \
    --checkpoint-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints" \
    --output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output" \
    --device cuda \
    --seed 42 \
    --debug

In [ ]:
# Train all models (base, large, lite) — use A100 GPU for this.
# Uncomment to run:
# !python src/train_pipeline.py \
#     --model all \
#     --dataset-dir /content/domain-shift-trustworthy-ai/dataset/data \
#     --checkpoint-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints" \
#     --output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output" \
#     --device cuda \
#     --seed 42 \
#     --debug

In [ ]:
# Optional: Train with hyperparameter grid search
# !python src/train_pipeline.py \
#     --model base \
#     --dataset-dir /content/domain-shift-trustworthy-ai/dataset/data \
#     --checkpoint-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints" \
#     --output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output" \
#     --device cuda \
#     --grid-search \
#     --seed 42 \
#     --debug

### 3a. Inspect Training History (after training)

Shows epoch-by-epoch train loss, validation loss, and F1 for every domain specialist.

In [ ]:
# ── OUTPUT of model_trainer.train() — epoch-level metrics per specialist ──────
import json, glob

CKPT_ROOT = "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints"
history_files = sorted(glob.glob(f"{CKPT_ROOT}/base/*/training_history.json"))

for hf in history_files:
    domain = hf.split("/")[-2]
    with open(hf) as f:
        h = json.load(f)

    print(f"\n── {domain} specialist ──")
    print(f"  {'Epoch':>5}  {'TrainLoss':>10}  {'ValLoss':>8}  {'ValMacroF1':>11}  {'F1-0':>6}  {'F1-1':>6}")
    print("  " + "-" * 57)
    epochs      = h.get("epochs",        range(1, len(h["train_loss"]) + 1))
    train_loss  = h["train_loss"]
    val_loss    = h["val_loss"]
    val_mf1     = h["val_macro_f1"]
    val_f1_0    = h["val_f1_0"]
    val_f1_1    = h["val_f1_1"]
    for i, ep in enumerate(epochs):
        print(f"  {ep:>5}  {train_loss[i]:>10.4f}  {val_loss[i]:>8.4f}  "
              f"{val_mf1[i]:>11.4f}  {val_f1_0[i]:>6.4f}  {val_f1_1[i]:>6.4f}")
    best_ep = val_mf1.index(max(val_mf1))
    print(f"  → best epoch {epochs[best_ep]}  Macro-F1={max(val_mf1):.4f}")

## 4. Model Evaluation

Run comprehensive evaluation including in-domain, cross-domain, and perturbation testing.

> `--debug` is passed so the pipeline prints before/after perturbation text pairs,
> per-domain metrics, domain-shift values, statistical test inputs and outputs.

In [ ]:
# Evaluate base model with full perturbation testing.
# --dataset-dir points to the CSV files inside the cloned GitHub repo.
# --train-output-dir must match --output-dir used in train_pipeline.py
# so evaluation reuses the exact same test splits the model never saw during training.
# --debug prints: perturbation pairs, per-domain metrics, domain-shift, stat-test outputs.
!python src/evaluate_pipeline.py \
    --model base \
    --checkpoint-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints" \
    --dataset-dir /content/domain-shift-trustworthy-ai/dataset/data \
    --output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results" \
    --train-output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output" \
    --device cuda \
    --seed 42 \
    --debug

In [ ]:
# Quick evaluation (skip perturbation testing for faster results)
# !python src/evaluate_pipeline.py \
#     --model base \
#     --checkpoint-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints" \
#     --dataset-dir /content/domain-shift-trustworthy-ai/dataset/data \
#     --output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results" \
#     --train-output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output" \
#     --device cuda \
#     --skip-perturbation \
#     --seed 42 \
#     --debug

In [ ]:
# Evaluate all models (if you trained all)
# !python src/evaluate_pipeline.py \
#     --model all \
#     --checkpoint-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints" \
#     --dataset-dir /content/domain-shift-trustworthy-ai/dataset/data \
#     --output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results" \
#     --train-output-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output" \
#     --device cuda \
#     --seed 42 \
#     --debug

### 4a. Inspect Perturbation — Before vs After Text

Loads the saved test split and applies all three perturbation levels inline so you
can see exactly what the model receives at each noise intensity.

In [ ]:
# ── INPUT: clean test text  |  OUTPUT: perturbed text per level ──────────────
import sys
sys.path.insert(0, "/content/domain-shift-trustworthy-ai/src")

import pandas as pd
from perturbation_engine import PerturbationEngine

SPLITS_DIR = "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output/data_splits"
engine = PerturbationEngine(random_seed=42)

# Pick 3 samples from the Technology test split for a concrete demonstration
test_df = pd.read_csv(f"{SPLITS_DIR}/technology/test.csv")
samples = test_df.sample(n=3, random_state=0)["text"].tolist()

SEP = "═" * 80
for i, original in enumerate(samples):
    low    = engine.apply_perturbation(original, "low")
    medium = engine.apply_perturbation(original, "medium")
    high   = engine.apply_perturbation(original, "high")

    # Compute simple char-change ratio
    def change_ratio(a, b):
        return sum(x != y for x, y in zip(a, b)) / max(len(a), 1)

    print(f"\n{SEP}")
    print(f"Sample {i+1}")
    print(f"{SEP}")
    print(f"  ORIGINAL  : {original}")
    print(f"  LOW       : {low}")
    print(f"             char-change = {change_ratio(original, low):.1%}")
    print(f"  MEDIUM    : {medium}")
    print(f"             char-change = {change_ratio(original, medium):.1%}")
    print(f"  HIGH      : {high}")
    print(f"             char-change = {change_ratio(original, high):.1%}")
print(SEP)

### 4b. Inspect In-Domain Evaluation — Inputs and Outputs

Loads the saved evaluation results and shows, for each domain specialist:
the test texts going in, the predicted labels coming out, and the per-domain metrics.

In [ ]:
# ── INPUT: test texts + true labels  |  OUTPUT: predictions + metrics ─────────
import json, pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results/base"
SPLITS_DIR  = "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output/data_splits"

with open(f"{RESULTS_DIR}/complete_evaluation.json") as f:
    ev = json.load(f)

SEP = "─" * 80
print("IN-DOMAIN EVALUATION — inputs and predicted outputs")
print(SEP)
print(f"{'Domain':<14} {'N':>5}  {'Acc':>7}  {'MacroF1':>8}  {'F1-CB':>7}  {'FP':>5}  {'FN':>5}")
print(SEP)

for domain, data in ev["in_domain"].items():
    m  = data["metrics"]
    cm = m.get("confusion_matrix", [[0,0],[0,0]])
    print(f"{domain:<14} {data['num_samples']:>5}  "
          f"{m['accuracy']:>7.4f}  {m['macro_f1']:>8.4f}  "
          f"{m.get('f1_1', m.get('f1_1', 0)):>7.4f}  "
          f"{cm[0][1]:>5}  {cm[1][0]:>5}")

# Show 4 concrete prediction examples for Education
print(f"\n── Sample predictions: Education specialist on Education test set ──")
edu_test = pd.read_csv(f"{SPLITS_DIR}/education/test.csv")
edu_data = ev["in_domain"].get("Education", {})
preds    = edu_data.get("predictions", [])
trues    = edu_data.get("true_labels", [])
texts    = edu_test["text"].tolist()

print(f"  {'True':>13}  {'Pred':>13}  Text")
print(f"  {'─'*13}  {'─'*13}  {'─'*50}")
count = 0
for txt, true, pred in zip(texts, trues, preds):
    label_t = "clickbait" if true == 1 else "non-clickbait"
    label_p = "clickbait" if pred == 1 else "non-clickbait"
    mark = "  ✗" if true != pred else ""
    print(f"  {label_t:>13}  {label_p:>13}  {str(txt)[:60]}{mark}")
    count += 1
    if count >= 6:
        break

### 4c. Inspect Cross-Domain Evaluation — Source Model on Target Texts

Shows which source-domain specialist was used, what target-domain texts it received,
and how its predictions compare to the target specialist's in-domain performance.

In [ ]:
# ── INPUT: source specialist + target test texts  |  OUTPUT: metrics + shift ──
import json, pandas as pd

RESULTS_DIR = "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results/base"
SPLITS_DIR  = "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output/data_splits"

with open(f"{RESULTS_DIR}/complete_evaluation.json") as f:
    ev = json.load(f)

# ── Cross-domain performance matrix ──
domains = ["Education", "Health", "Politics", "Sport", "Technology"]
cd = ev["cross_domain"]

print("CROSS-DOMAIN Macro-F1  (row = source specialist, col = target test set)")
print(f"{'Source':<12}", end="")
for d in domains:
    print(f"  {d[:9]:>9}", end="")
print()
print("-" * 65)
for src in domains:
    print(f"{src:<12}", end="")
    for tgt in domains:
        key = f"{src}->{tgt}"
        f1  = cd.get(key, {}).get("metrics", {}).get("macro_f1", 0.0)
        marker = "◆" if src == tgt else " "
        print(f"  {f1:>8.4f}{marker}", end="")
    print()

# ── Concrete example: Technology specialist predicting Health test set ──
print("\n── Example: Technology specialist applied to Health test texts ──")
key = "Technology->Health"
cd_data = cd.get(key, {})
preds   = cd_data.get("predictions", [])
trues   = cd_data.get("true_labels", [])
shift   = cd_data.get("domain_shift", {})

health_test = pd.read_csv(f"{SPLITS_DIR}/health/test.csv")
texts       = health_test["text"].tolist()

mf1_src = ev["in_domain"].get("Technology", {}).get("metrics", {}).get("macro_f1", 0)
mf1_tgt = ev["in_domain"].get("Health",     {}).get("metrics", {}).get("macro_f1", 0)
mf1_cd  = cd_data.get("metrics", {}).get("macro_f1", 0)
print(f"  Technology in-domain F1 : {mf1_src:.4f}")
print(f"  Health in-domain F1     : {mf1_tgt:.4f}")
print(f"  Technology→Health F1    : {mf1_cd:.4f}")
print(f"  Source Drop (SD)        : {shift.get('sd_macro_f1', 0):+.4f}")
print(f"  Target Drop (TD)        : {shift.get('td_macro_f1', 0):+.4f}")

print(f"\n  {'True':>13}  {'Pred':>13}  Health headline")
print(f"  {'─'*13}  {'─'*13}  {'─'*50}")
count = 0
for txt, true, pred in zip(texts, trues, preds):
    label_t = "clickbait" if true == 1 else "non-clickbait"
    label_p = "clickbait" if pred == 1 else "non-clickbait"
    mark = "  ✗" if true != pred else ""
    print(f"  {label_t:>13}  {label_p:>13}  {str(txt)[:55]}{mark}")
    count += 1
    if count >= 6:
        break

## 5. View Results

In [ ]:
# Print full evaluation summary + error analysis
# --results-dir must point to the model subfolder (results/base, results/large, etc.)
# because evaluate_pipeline.py saves complete_evaluation.json inside results/{model}/
!python src/print_evaluation_summary.py \
    --results-dir "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results/base" \
    --model BASE

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_base = "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results/base"

# Perturbation per domain
img = mpimg.imread(f"{results_base}/perturbation_per_domain.png")
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Perturbation per Domain")
plt.show()

# Mean perturbation
img = mpimg.imread(f"{results_base}/perturbation_mean.png")
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Mean Perturbation")
plt.show()

In [ ]:
# View training summary
import json

with open('/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/output/training_summary.json', 'r') as f:
    training_summary = json.load(f)

print("Training Summary:")
print(f"Total samples: {training_summary['total_samples']}")
print(f"Domains: {', '.join(training_summary['domains'])}")
print(f"\nBest Macro-F1 per model per domain:")
for model_name, domain_results in training_summary['training_results'].items():
    print(f"  {model_name.upper()}:")
    for domain_name, info in domain_results.items():
        print(f"    {domain_name:<14} epoch={info['best_epoch']}  "
              f"macro_f1={info['best_macro_f1']:.4f}  "
              f"f1_0={info['best_f1_0']:.4f}  f1_1={info['best_f1_1']:.4f}")

In [ ]:
# View evaluation summary
with open('/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results/evaluation_summary.json', 'r') as f:
    eval_summary = json.load(f)

print("Evaluation Summary:")
for model, metrics in eval_summary['model_summaries'].items():
    print(f"\n{model.upper()}:")
    print(f"  Avg In-Domain F1:     {metrics['avg_in_domain_f1']:.4f}")
    print(f"  Avg Cross-Domain F1:  {metrics['avg_cross_domain_f1']:.4f}")
    if 'avg_perturbation_f1' in metrics:
        print(f"  Avg Perturbation F1:  {metrics['avg_perturbation_f1']:.4f}")

In [ ]:
# Visualize results
import matplotlib.pyplot as plt
import seaborn as sns

models = list(eval_summary['model_summaries'].keys())
in_domain_f1    = [eval_summary['model_summaries'][m]['avg_in_domain_f1']    for m in models]
cross_domain_f1 = [eval_summary['model_summaries'][m]['avg_cross_domain_f1'] for m in models]

x = range(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([i - width/2 for i in x], in_domain_f1,    width, label='In-Domain',    alpha=0.8)
ax.bar([i + width/2 for i in x], cross_domain_f1, width, label='Cross-Domain', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('F1 Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(
    '/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results/performance_comparison.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

## 6. Launch Dashboard (Optional)

Launch the interactive Streamlit dashboard for visualization and testing.

In [ ]:
# Install localtunnel for public URL
!npm install -g localtunnel

In [ ]:
# Launch dashboard in background
import subprocess, threading, time

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'src/dashboard_app.py', '--server.port', '8501'])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()

time.sleep(5)
print("Streamlit is starting...")

In [ ]:
# Create public URL with localtunnel
!lt --port 8501

## 7. Download Results

In [ ]:
# Create zip file of all results
!zip -r results.zip "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/results/"
!zip -r checkpoints.zip "/content/drive/MyDrive/BINUS/Prethesis and Thesis/experiment/checkpoints/"

print("Results and checkpoints zipped successfully!")
print("Files are saved in Google Drive and can be downloaded from there.")